# Operationalizing an AWS ML Project — Step 1 notebook

Run this notebook on a **SageMaker notebook instance** (kernel `conda_python3`), as described in Step 1 of the project.
It also runs in a SageMaker Studio JupyterLab space.

## Install the SageMaker Python SDK v2

This notebook uses SageMaker Python SDK **v2** (`sagemaker.tuner`, `sagemaker.pytorch`, `sagemaker.debugger`).
SDK **v3** removed those modules, so run the next cell first on every fresh kernel.

- If it installs a different version, **restart the kernel** (**Kernel → Restart Kernel**), then continue with the cell after it.
- The cell after it prints the SDK version (`2.x`). Two `sagemaker.config INFO - Not applying SDK defaults…` lines are normal.
- SDK v2 is on AWS's deprecation path, so the version-check cell switches off its repeated deprecation warning; this project is
  written for v2 and works with it. In SageMaker Studio you may also see pip "dependency resolver" warnings — harmless here.

In [1]:
%pip install --quiet "sagemaker>=2.231,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"  # this project targets SDK v2 on purpose; see the note above

import sagemaker

version = getattr(sagemaker, "__version__", "3.x")
print("SageMaker Python SDK", version)
assert version.startswith("2."), "SDK v3 is still loaded: restart the kernel (Kernel > Restart Kernel), then run this cell again."

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
SageMaker Python SDK 2.257.6


## Set Your S3 Bucket

Set `bucket_name` to the S3 bucket you created in Step 1. Use a name that contains **`sagemaker`**
(for example `sagemaker-dog-breeds-48213`). The role created with your notebook instance can use any bucket, but roles that
rely only on the AWS managed `AmazonSageMakerFullAccess` policy can only read and write buckets whose name contains `sagemaker`.

In [3]:
bucket_name = "sagemaker-dog-breeds-48213"

assert bucket_name, "Set bucket_name to the name of your S3 bucket."
if "sagemaker" not in bucket_name.lower():
    print("NOTE: the bucket name does not contain 'sagemaker'. That works with the role created with a notebook instance,")
    print("but other SageMaker roles may get AccessDenied - see the Step 1 instructions.")

## Get the Data and Copy It to S3

The dataset is about 1.1 GB. **You only need to copy it to S3 once.** Run the check below first:
if it says the data is already in your bucket, skip the three cells after it and continue at **Import**.

In [4]:
import subprocess

listing = subprocess.run(["aws", "s3", "ls", f"s3://{bucket_name}/train/"], capture_output=True, text=True).stdout
if listing.strip():
    print(f"Data is already in s3://{bucket_name}/ - skip the next three cells.")
else:
    print(f"No data in s3://{bucket_name}/ yet - run the next three cells.")

Data is already in s3://sagemaker-dog-breeds-48213/ - skip the next three cells.


In [ ]:
%%capture
!wget -c https://s3-us-west-1.amazonaws.com/udacity-aind/dog-project/dogImages.zip

In [ ]:
%%capture
!unzip -o -q dogImages.zip

In [ ]:
%%capture
!aws s3 cp dogImages s3://{bucket_name}/ --recursive

## Import

In [4]:
import sagemaker
import boto3
from sagemaker.tuner import CategoricalParameter, ContinuousParameter, HyperparameterTuner
from sagemaker.pytorch import PyTorch
from sagemaker.debugger import Rule, DebuggerHookConfig, TensorBoardOutputConfig, CollectionConfig, ProfilerRule, rule_configs
from sagemaker.debugger import ProfilerConfig
import os

## Set Up Parameters, Estimator, and Tuner

In [5]:
hyperparameter_ranges = {
    "learning_rate": ContinuousParameter(0.001, 0.1),
    "batch_size": CategoricalParameter([32, 64, 128, 256, 512]),
}

role = "arn:aws:iam::100774010047:role/service-role/AmazonSageMaker-ExecutionRole-20261009T011010"

objective_metric_name = "Test Loss"
objective_type = "Minimize"
metric_definitions = [{"Name": "Test Loss", "Regex": "Testing Loss: ([0-9\\.]+)"}]

In [6]:
estimator = PyTorch(
    entry_point="hpo.py",
    base_job_name='pytorch_dog_hpo',
    role=role,
    framework_version="2.2",
    instance_count=1,
    instance_type="ml.m5.xlarge",
    py_version='py310'
)

tuner = HyperparameterTuner(
    estimator,
    objective_metric_name,
    hyperparameter_ranges,
    metric_definitions,
    max_jobs=2,
    max_parallel_jobs=2,
    objective_type=objective_type
)

## Fit the Tuner

In [7]:
os.environ["SM_CHANNEL_TRAINING"] = f"s3://{bucket_name}/"
os.environ["SM_MODEL_DIR"] = f"s3://{bucket_name}/model/"
os.environ["SM_OUTPUT_DATA_DIR"] = f"s3://{bucket_name}/output/"
tuner.fit({"training": f"s3://{bucket_name}/"})

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


...........................................................................................................................................................................................................................................................................................................!


## Describe the Tuning Results

In [8]:
import boto3
from sagemaker.analytics import HyperparameterTuningJobAnalytics

# Settings (change region/keyword if needed)
REGION = boto3.session.Session().region_name   # e.g., "us-east-1"
NAME_KEYWORD = "pytorch-training"              # keyword used for listing

sm = boto3.client("sagemaker", region_name=REGION)

# 1) Find the most recent HPO job that matches the keyword
resp = sm.list_hyper_parameter_tuning_jobs(
    NameContains=NAME_KEYWORD,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

summaries = resp.get("HyperParameterTuningJobSummaries", [])
assert summaries, f"No HPO jobs found with keyword '{NAME_KEYWORD}' in region '{REGION}'."

tuning_job_name = summaries[0]["HyperParameterTuningJobName"]
print("Using HPO job:", tuning_job_name)

# 2) Analyze the selected job
exp = HyperparameterTuningJobAnalytics(hyperparameter_tuning_job_name=tuning_job_name)
jobs = exp.dataframe()

# 3) Sort by objective value in descending order (only if the column exists)
if "FinalObjectiveValue" in jobs.columns:
    jobs = jobs.sort_values("FinalObjectiveValue", ascending=False)

jobs.head()  # preview

Using HPO job: pytorch-training-261010-1712


,batch_size,learning_rate,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
0,"""128""",0.084586,pytorch-training-261010-1712-002-a5a464af,Completed,541.0,2026-10-10 17:12:58+00:00,2026-10-10 17:33:42+00:00,1244.0
1,"""32""",0.001097,pytorch-training-261010-1712-001-a3ccb019,Completed,33.0,2026-10-10 17:13:01+00:00,2026-10-10 17:30:44+00:00,1063.0


## Important: If Kernel Dies, How to Continue From a Completed Training Job

In [28]:
BetterTrainingJobName='pytorch-training-210623-2156-001-fdd5e081'

In [29]:
my_estimator = sagemaker.estimator.Estimator.attach(BetterTrainingJobName)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 my_estimator = sagemaker.estimator.Estimator.attach(BetterTrainingJobName)                   │
│   2                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:1589   │
│ in attach                                                                                        │
│                                                                                                  │
│   1586 │   │   │   Instance of the calling ``Estimator`` Class with the attached                 │
│   1587 │   │   │   training job.                                                                 │
│   1588 │   │   """                                                                               │
│ ❱ 1589 │   │   return cls._attach(                                                               │
│   1590 │   │   │   training_job_name=training_job_name,                                          │
│   1591 │   │   │   sagemaker_session=sagemaker_session,                                          │
│   1592 │   │   │   model_channel_name=model_channel_name,                                        │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:1609   │
│ in _attach                                                                                       │
│                                                                                                  │
│   1606 │   │   """                                                                               │
│   1607 │   │   sagemaker_session = sagemaker_session or Session()                                │
│   1608 │   │                                                                                     │
│ ❱ 1609 │   │   job_details = sagemaker_session.sagemaker_client.describe_training_job(           │
│   1610 │   │   │   TrainingJobName=training_job_name                                             │
│   1611 │   │   )                                                                                 │
│   1612 │   │   init_params = cls._prepare_init_params_from_job_description(job_details, model_c  │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/client.py:606 in     │
│ _api_call                                                                                        │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-p

In [ ]:
my_estimator.hyperparameters()

In [ ]:
best_estimator = my_estimator

## Prepare to Perform Training on Best Estimator

In [9]:
best_estimator = tuner.best_estimator()


2026-10-10 17:30:45 Starting - Preparing the instances for training
2026-10-10 17:30:45 Downloading - Downloading the training image
2026-10-10 17:30:45 Training - Training image download completed. Training in progress.
2026-10-10 17:30:45 Uploading - Uploading generated training model
2026-10-10 17:30:45 Completed - Resource released due to keep alive period expiry


In [10]:
best_estimator.hyperparameters()

{'_tuning_objective_metric': '"Test Loss"',
 'batch_size': '"32"',
 'learning_rate': '0.001097368851516355',
 'sagemaker_container_log_level': '20',
 'sagemaker_estimator_class_name': '"PyTorch"',
 'sagemaker_estimator_module': '"sagemaker.pytorch.estimator"',
 'sagemaker_job_name': '"pytorch_dog_hpo-2026-10-10-17-12-08-640"',
 'sagemaker_program': '"hpo.py"',
 'sagemaker_region': '"us-east-1"',
 'sagemaker_submit_directory': '"s3://sagemaker-us-east-1-100774010047/pytorch_dog_hpo-2026-10-10-17-12-08-640/source/sourcedir.tar.gz"'}

In [11]:
hyperparameters = {"batch_size": int(best_estimator.hyperparameters()['batch_size'].replace('"', '')), \
                   "learning_rate": best_estimator.hyperparameters()['learning_rate']}
hyperparameters

{'batch_size': 32, 'learning_rate': '0.001097368851516355'}

In [12]:
rules = [
    Rule.sagemaker(rule_configs.vanishing_gradient()),
    Rule.sagemaker(rule_configs.overfit()),
    Rule.sagemaker(rule_configs.overtraining()),
    Rule.sagemaker(rule_configs.poor_weight_initialization()),
    ProfilerRule.sagemaker(rule_configs.ProfilerReport()),
]

In [13]:
hook_config = DebuggerHookConfig(
    hook_parameters={
        "train.save_interval": "1",
        "eval.save_interval": "1"
    }
)

profiler_config = ProfilerConfig(
    system_monitor_interval_millis=500
)

## Creating an Estimator

In [36]:
# Single-instance estimator. Leave this cell as it is: multi-instance training
# goes in its own section below.
estimator = PyTorch(
    entry_point='hpo.py',
    base_job_name='dog-pytorch',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='2.2',
    py_version='py310',
    hyperparameters=hyperparameters,
    ## Debugger and Profiler parameters
    rules=rules,
    debugger_hook_config=hook_config,
    profiler_config=profiler_config,
)

In [37]:
estimator.fit({"training": f"s3://{bucket_name}/"}, wait=True)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: dog-pytorch-2026-10-09-15-59-31-358


2026-10-09 15:59:33 Starting - Starting the training job...
2026-10-09 16:00:04 Starting - Preparing the instances for trainingVanishingGradient: InProgress
Overfit: InProgress
Overtraining: InProgress
PoorWeightInitialization: InProgress
ProfilerReport: InProgress
...
2026-10-09 16:00:24 Downloading - Downloading input data......
2026-10-09 16:01:34 Downloading - Downloading the training image...
2026-10-09 16:02:05 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning:

<h2>Creating an Estimator - Multi-Instance Training</h2>

Do this section **after** you have deployed and tested the single-instance endpoint below.

1. In the next cell, create a new `PyTorch` estimator with `instance_count` greater than 1 (copy the estimator above and change it).
2. Assign it to the variable **`estimator`** and fit it with `wait=True`, so that the **Deployment** cells deploy the multi-instance model.
3. Re-run the cells from here to the end of **Deployment**. You will then have two endpoints: use the **multi-instance** one in Steps 3–5 and delete the single-instance one.

In [14]:
### In this cell, create and fit an estimator using multi-instance training.
estimator = PyTorch(
    entry_point='hpo.py',
    base_job_name='dog-pytorch-multi',
    role=role,
    instance_count=2,
    instance_type='ml.m5.xlarge',
    framework_version='2.2',
    py_version='py310',
    hyperparameters=hyperparameters,
    ## Debugger and Profiler parameters
    rules=rules,
    debugger_hook_config=hook_config,
    profiler_config=profiler_config,
)


In [15]:
estimator.fit({"training": f"s3://{bucket_name}/"}, wait=True)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: dog-pytorch-multi-2026-10-10-17-45-51-891


2026-10-10 17:45:55 Starting - Starting the training job...
2026-10-10 17:46:25 Starting - Preparing the instances for trainingVanishingGradient: InProgress
Overfit: InProgress
Overtraining: InProgress
PoorWeightInitialization: InProgress
ProfilerReport: InProgress
...............
2026-10-10 17:48:57 Downloading - Downloading input data......
2026-10-10 17:49:57 Downloading - Downloading the training image...
2026-10-10 17:50:26 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecat

<h2>Deployment</h2>

In [16]:
# The most recently fitted `estimator`: single-instance on the first run,
# multi-instance after you run the section above.
model_location = estimator.model_data

In [17]:
import sagemaker
import boto3
from sagemaker.tuner import CategoricalParameter, ContinuousParameter, HyperparameterTuner
from sagemaker.pytorch import PyTorch
from sagemaker import get_execution_role
from sagemaker.debugger import Rule, DebuggerHookConfig, TensorBoardOutputConfig, CollectionConfig, ProfilerRule, rule_configs
from sagemaker.debugger import ProfilerConfig

from sagemaker.pytorch import PyTorchModel
from sagemaker.predictor import Predictor

In [18]:
jpeg_serializer = sagemaker.serializers.IdentitySerializer("image/jpeg")
json_deserializer = sagemaker.deserializers.JSONDeserializer()


class ImagePredictor(Predictor):
    def __init__(self, endpoint_name, sagemaker_session):
        super(ImagePredictor, self).__init__(
            endpoint_name,
            sagemaker_session=sagemaker_session,
            serializer=jpeg_serializer,
            deserializer=json_deserializer,
        )

In [19]:
pytorch_model = PyTorchModel(model_data=model_location, role=role, entry_point='inference.py', py_version='py310',
                             framework_version='2.2',
                             predictor_cls=ImagePredictor)

In [20]:
predictor = pytorch_model.deploy(initial_instance_count=1, instance_type='ml.m5.large')


INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-100774010047/dog-pytorch-multi-2026-10-10-17-45-51-891/output/model.tar.gz), script artifact (None), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-100774010047/pytorch-inference-2026-10-10-18-07-10-385/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: pytorch-inference-2026-10-10-18-07-19-412
INFO:sagemaker:Creating endpoint-config with name pytorch-inference-2026-10-10-18-07-19-802
INFO:sagemaker:Creating endpoint with name pytorch-inference-2026-10-10-18-07-19-802


---

ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/urllib3/connection.py:204 in  │
│ _new_conn                                                                                        │
│                                                                                                  │
│    201 │   │   :return: New socket connection.                                                   │
│    202 │   │   """                                                                               │
│    203 │   │   try:                                                                              │
│ ❱  204 │   │   │   sock = connection.create_connection(                                          │
│    205 │   │   │   │   (self._dns_host, self.port),                                              │
│    206 │   │   │   │   self.timeout,                                                             │
│    207 │   │   │   │   source_address=self.source_address,                                       │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/urllib3/util/connection.py:60 │
│ in create_connection                                                                             │
│                                                                                                  │
│    57 │   except UnicodeError:                                                                   │
│    58 │   │   raise LocationParseError(f"'{host}', label empty or too long") from None           │
│    59 │                                                                                          │
│ ❱  60 │   for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):                 │
│    61 │   │   af, socktype, proto, canonname, sa = res                                           │
│    62 │   │   sock = None                                                                        │
│    63 │   │   try:                                                                               │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/socket.py:967 in getaddrinfo                │
│                                                                                                  │
│   964 │   # We override this function since we want to translate the numeric family              │
│   965 │   # and socket type values to enum constants.                                            │
│   966 │   addrlist = []                                                                          │
│ ❱ 967 │   for res in _socket.getaddrinfo(host, port, family, type, proto, flags):                │
│   968 │   │   af, socktype, proto, canonname, sa = res                                           │
│   969 │   │   addrlist.append((_intenum_converter(af, AddressFamily),                            │
│   970 │   │   │   │   │   │    _intenum_converter(socktype, SocketKind),                         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
gaierror: [Errno -2] Name or service not known

The above exception was the direct cause of the following exception:

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/httpsession.py:509   │
│ in send                                                                                          │
│                                                                                                  │
│   506 │   │   │   │   # timeout=None to urlopen would disable timeouts entirely                  │
│   507 │   │   │   │   # rather than defer 

In [ ]:
print("Endpoint name:", predictor.endpoint_name)

In [ ]:
import requests
#request_dict = { "url": "https://cdn1-www.cattime.com/assets/uploads/2011/12/file_2744_british-shorthair-460x290-460x290.jpg" }
request_dict = { "url": "https://s3.amazonaws.com/cdn-origin-etr.akc.org/wp-content/uploads/2017/11/20113314/Carolina-Dog-standing-outdoors.jpg" }

img_bytes = requests.get(request_dict['url']).content
type(img_bytes)

In [ ]:
from PIL import Image
import io
Image.open(io.BytesIO(img_bytes))

In [ ]:
response = predictor.predict(img_bytes, initial_args={"ContentType": "image/jpeg"})

In [ ]:
import json
response2 = predictor.predict(json.dumps(request_dict), initial_args={"ContentType": "application/json"})

In [ ]:
type(response2[0][0])

In [ ]:
response2[0]

In [ ]:
import numpy as np
np.argmax(response, 1)


In [ ]:
print(len(response[0]), "class scores")  # one score per dog breed: 133

## Step 6: Clean Up

**Use the next cell only after you have finished Steps 2–5 and taken all of your screenshots** — your Lambda function and
auto-scaling in Steps 3–5 use the endpoint. It does nothing until you set `CLEAN_UP = True`, so "Run All" is safe.

It removes auto-scaling from, and then deletes, every endpoint this notebook deployed, with its endpoint configuration and model.
It does **not** delete your S3 buckets, EC2 instance, Lambda function or this notebook instance: follow the **Step 6: Clean Up** checklist in the classroom for those.

In [ ]:
CLEAN_UP = False  # set to True only when you have finished the whole project

import boto3

sm = boto3.client("sagemaker")
autoscaling = boto3.client("application-autoscaling")

endpoints = sm.list_endpoints(NameContains="pytorch-inference", MaxResults=100)["Endpoints"] if CLEAN_UP else []
if not CLEAN_UP:
    print("Nothing deleted. Set CLEAN_UP = True in this cell when you have finished Steps 2-5.")

for ep in endpoints:
    name = ep["EndpointName"]
    config = sm.describe_endpoint(EndpointName=name)["EndpointConfigName"]
    variants = sm.describe_endpoint_config(EndpointConfigName=config)["ProductionVariants"]
    for variant in variants:
        resource_id = f"endpoint/{name}/variant/{variant['VariantName']}"
        try:
            autoscaling.deregister_scalable_target(ServiceNamespace="sagemaker", ResourceId=resource_id,
                                                   ScalableDimension="sagemaker:variant:DesiredInstanceCount")
            print("removed auto-scaling from", resource_id)
        except autoscaling.exceptions.ObjectNotFoundException:
            pass
    sm.delete_endpoint(EndpointName=name)
    sm.delete_endpoint_config(EndpointConfigName=config)
    for variant in variants:
        sm.delete_model(ModelName=variant["ModelName"])
    print("deleted endpoint", name)

print("Remaining endpoints:", [e["EndpointName"] for e in sm.list_endpoints()["Endpoints"]])